# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aanvi02/ML-work/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

I built a feature vector using only information that is available before making a refresh decision. Numerical features are kept as numeric values, categorical features are encoded, and missing values are filled where required. Label-derived and private information are excluded to prevent data leakage and protect privacy.

In [3]:
!git clone https://github.com/Aanvi02/ML-work.git

Cloning into 'ML-work'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 134 (delta 47), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.85 MiB | 15.69 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [4]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/ML-work/data/raw/content_refresh_anonymized.csv")

# Selected features
features = [
    "search_volume",
    "competition",
    "ctr",
    "avg_position",
    "word_count",
    "content_age_days",
    "days_since_last_update"
]

categorical = [
    "content_type",
    "main_intent"
]

# Feature dataframe
X = df[features + categorical].copy()

# Fill missing values
for col in features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical:
    X[col] = X[col].fillna("Unknown")

# One-hot encode categorical columns
X = pd.get_dummies(X, columns=categorical, drop_first=True)

print("Feature vector shape:", X.shape)
X.head()

Feature vector shape: (30000, 13)


,search_volume,competition,ctr,avg_position,word_count,content_age_days,days_since_last_update,content_type_feedly article,content_type_keyword article,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional
0,10.0,0.67,0.76,10.6,3221.0,187,20,False,True,False,False,False,True
1,90.0,0.01,0.05,20.3,2481.0,445,25,False,True,False,True,False,False
2,0.0,0.00,0.09,36.5,3515.0,141,20,False,True,False,True,False,False
3,10.0,0.00,0.49,6.2,2877.0,463,22,False,True,True,False,False,False
4,0.0,0.00,0.13,44.0,2803.0,263,14,False,True,False,True,False,False


### Feature Notes

- **search_volume:** Represents the estimated monthly search demand for a topic. Missing values are filled with the median. It is available before making a prediction.
- **competition:** Indicates how competitive a keyword is. Missing values are filled with the median. It is available before prediction.
- **ctr:** Historical click-through rate of the content. Missing values are filled with the median. It is available before prediction because it comes from past performance.
- **avg_position:** Average search ranking position. Missing values are filled with the median. It is available before prediction.
- **word_count:** Number of words in the content. Missing values are filled with the median. It is known before making a refresh decision.
- **content_age_days:** Age of the content in days. Missing values are filled with the median. It is available before prediction.
- **days_since_last_update:** Number of days since the content was last updated. Missing values are filled with the median. It is available before prediction.
- **content_type:** Category of the content. Missing values are replaced with "Unknown". It is available before prediction.
- **main_intent:** Primary search intent of the content. Missing values are replaced with "Unknown". It is available before prediction.

In [5]:
# Selected features
selected_features = [
    "search_volume",
    "competition",
    "ctr",
    "avg_position",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "content_type",
    "main_intent"
]

# Missing values
print("Missing values:")
print(df[selected_features].isnull().sum())

print("\nData types:")
print(df[selected_features].dtypes)


Missing values:
search_volume             2468
competition               2468
ctr                          0
avg_position                 0
word_count                7699
content_age_days             0
days_since_last_update       0
content_type                 0
main_intent               2374
dtype: int64

Data types:
search_volume             float64
competition               float64
ctr                       float64
avg_position              float64
word_count                float64
content_age_days            int64
days_since_last_update      int64
content_type               object
main_intent                object
dtype: object


### Leakage Hunt

I checked my selected features for potential data leakage. I excluded any feature that directly contains future information or is derived from the target label. Features such as `trend_direction` and `trend_pct` are treated as labels and are not used as model inputs. I also avoided using future-window information when building the feature vector. The remaining features are available before making a refresh decision and are safe for modelling.

In [6]:
# Potential leakage columns
leakage_columns = [
    "trend_direction",
    "trend_pct"
]

print("Potential leakage columns found:")

for col in leakage_columns:
    if col in df.columns:
        print(f"✓ {col}")
    else:
        print(f"✗ {col} not present")

# Features used for modelling
safe_features = [
    "search_volume",
    "competition",
    "ctr",
    "avg_position",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "content_type",
    "main_intent"
]

print("\nSafe modelling features:")
print(safe_features)


Potential leakage columns found:
✓ trend_direction
✓ trend_pct

Safe modelling features:
['search_volume', 'competition', 'ctr', 'avg_position', 'word_count', 'content_age_days', 'days_since_last_update', 'content_type', 'main_intent']


### What I Excluded and Why

- **content_id** – Unique identifier only; it does not help the model learn patterns.
- **client_id** – Excluded to avoid client-specific information and improve generalization.
- **trend_direction** – Used as the target label, so using it as a feature would cause data leakage.
- **trend_pct** – Derived from future performance and may leak target information.
- **impressions_last_30d, clicks_last_30d, sessions_last_30d** – Not used because they can overlap with the prediction window and may introduce leakage.
- **impressions_prev_30d, clicks_prev_30d, sessions_prev_30d** – Excluded to keep the feature set simple for the baseline model.

In [7]:
excluded_features = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print("Excluded features:")
for feature in excluded_features:
    if feature in df.columns:
        print(f"✓ {feature}")


Excluded features:
✓ content_id
✓ client_id
✓ trend_direction
✓ trend_pct
✓ impressions_last_30d
✓ clicks_last_30d
✓ sessions_last_30d
✓ impressions_prev_30d
✓ clicks_prev_30d
✓ sessions_prev_30d


## Self-check

- [x] Every section above is filled — markdown thinking and supporting code.
- [x] The notebook runs from top to bottom without errors (Runtime → Run all).
- [x] No client names, URLs, or private queries are included.
- [x] My claims use careful language such as observed, measured, directional, and decision-support.
- [x] The notebook is committed to my repository under `work/notebooks/`.